# Mono-First Stereo Calibration

This notebook calibrates the two Basler cameras in three stages:

1. left camera mono calibration
2. right camera mono calibration
3. stereo calibration with fixed mono intrinsics

Keep camera order, exposure, gain, focus, resolution, mounting, board geometry, and right-image orientation unchanged throughout the session.


In [ ]:
from __future__ import annotations

import glob
import os
import pickle
import time
from dataclasses import dataclass
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

try:
    from pypylon import pylon
except Exception as exc:
    pylon = None
    print(f'pypylon unavailable: {exc}')

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['image.cmap'] = 'gray'

## Configuration

Check these values against the printed ChArUco board before taking new images. The square and marker sizes are given in millimeters.


In [ ]:
CALIBRATION_TARGET = 'charuco'  # 'charuco' or 'chessboard'

# ChArUco board geometry.
CHARUCO_SQUARES = (11, 8)       # squares in x/y direction, not inner corners
CHARUCO_SQUARE_SIZE = 10.0      # millimeters
CHARUCO_MARKER_SIZE = 7.0       # millimeters
ARUCO_DICT_NAME = 'DICT_4X4_50' # use DICT_4X4_100/250/1000 if your printed board says so
CHARUCO_LEGACY_PATTERN = True   # True for boards generated with OpenCV's legacy ChArUco layout
MIN_CHARUCO_CORNERS = 12        # minimum detected/interpolated ChArUco corners per image

# Derived inner-corner count used by shared calibration helpers.
BOARD_SIZE = (CHARUCO_SQUARES[0] - 1, CHARUCO_SQUARES[1] - 1)  # ChArUco chessboard corners: 10 x 7
SQUARE_SIZE = CHARUCO_SQUARE_SIZE

LEFT_CAMERA_INDEX = 0
RIGHT_CAMERA_INDEX = 1
ROTATE_RIGHT_180 = False  # image rotation during capture

# Stereo solve configuration. ChArUco marker detection expects the saved image orientation.
SELECTED_RIGHT_IMAGE_VARIANT = 'as_saved'
SELECTED_REVERSE_RIGHT_CORNERS = False
SELECTED_SWAPPED_INTRINSICS = True

# Paths work from the project root and from inside ./calib.
CALIB_ROOT = Path('calib') if Path('calib').is_dir() else Path('.')
PROJECT_ROOT = CALIB_ROOT.parent if CALIB_ROOT.name == 'calib' else Path('..')

MONO_LEFT_DIR = CALIB_ROOT / 'mono_calib_left'
MONO_RIGHT_DIR = CALIB_ROOT / 'mono_calib_right'
STEREO_CALIB_DIR = CALIB_ROOT / 'stereo_calib_images'

MONO_PARAMS_FILE = CALIB_ROOT / 'mono_camera_params.pkl'
STEREO_PARAMS_FILE = CALIB_ROOT / 'stereo_params_mono_first.pkl'

MIN_MONO_IMAGES = 12
MIN_STEREO_PAIRS = 12

CAPTURE_DIR = PROJECT_ROOT / 'captures'

for directory in [MONO_LEFT_DIR, MONO_RIGHT_DIR, STEREO_CALIB_DIR, CAPTURE_DIR]:
    directory.mkdir(exist_ok=True)

print('Calibration target:', CALIBRATION_TARGET)
print('ChArUco squares X/Y:', CHARUCO_SQUARES)
print('ChArUco square size [mm]:', CHARUCO_SQUARE_SIZE)
print('ChArUco marker size [mm]:', CHARUCO_MARKER_SIZE)
print('ArUco dictionary:', ARUCO_DICT_NAME)
print('ChArUco legacy pattern:', CHARUCO_LEGACY_PATTERN)
print('Selected stereo config:', SELECTED_RIGHT_IMAGE_VARIANT, SELECTED_REVERSE_RIGHT_CORNERS, SELECTED_SWAPPED_INTRINSICS)
print('Mono folders:', MONO_LEFT_DIR, MONO_RIGHT_DIR)
print('Stereo folder:', STEREO_CALIB_DIR)


## Shared Calibration Helpers

In [ ]:
@dataclass
class MonoCalibration:
    rms: float
    camera_matrix: np.ndarray
    distortion: np.ndarray
    rvecs: list
    tvecs: list
    image_size: tuple[int, int]
    valid_files: list[str]


def read_gray(path: str | Path) -> np.ndarray | None:
    return cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)


def require_aruco():
    if not hasattr(cv2, 'aruco'):
        raise RuntimeError(
            'cv2.aruco is unavailable. Install opencv-contrib-python in this environment to use ChArUco calibration.'
        )


def get_aruco_dictionary():
    require_aruco()
    dictionary_id = getattr(cv2.aruco, ARUCO_DICT_NAME)
    if hasattr(cv2.aruco, 'getPredefinedDictionary'):
        return cv2.aruco.getPredefinedDictionary(dictionary_id)
    return cv2.aruco.Dictionary_get(dictionary_id)


def get_charuco_board():
    require_aruco()
    dictionary = get_aruco_dictionary()
    if hasattr(cv2.aruco, 'CharucoBoard'):
        board = cv2.aruco.CharucoBoard(CHARUCO_SQUARES, CHARUCO_SQUARE_SIZE, CHARUCO_MARKER_SIZE, dictionary)
    else:
        board = cv2.aruco.CharucoBoard_create(
            CHARUCO_SQUARES[0],
            CHARUCO_SQUARES[1],
            CHARUCO_SQUARE_SIZE,
            CHARUCO_MARKER_SIZE,
            dictionary,
        )
    if hasattr(board, 'setLegacyPattern'):
        board.setLegacyPattern(CHARUCO_LEGACY_PATTERN)
    return board


def get_charuco_object_corners() -> np.ndarray:
    board = get_charuco_board()
    if hasattr(board, 'getChessboardCorners'):
        corners = board.getChessboardCorners()
    else:
        corners = board.chessboardCorners
    return np.asarray(corners, dtype=np.float32)


def make_object_points(board_size=BOARD_SIZE, square_size=SQUARE_SIZE) -> np.ndarray:
    if CALIBRATION_TARGET == 'charuco':
        return get_charuco_object_corners()
    objp = np.zeros((board_size[0] * board_size[1], 3), np.float32)
    objp[:, :2] = np.mgrid[0:board_size[0], 0:board_size[1]].T.reshape(-1, 2)
    objp[:, :2] *= square_size
    return objp


def detect_charuco_corners(gray: np.ndarray):
    board = get_charuco_board()
    dictionary = get_aruco_dictionary()

    # OpenCV's ArUco/ChArUco API changed across versions. Newer builds expose
    # CharucoDetector.detectBoard; older opencv-contrib builds expose
    # interpolateCornersCharuco. Support both so the notebook survives either one.
    if hasattr(cv2.aruco, 'CharucoDetector'):
        detector = cv2.aruco.CharucoDetector(board)
        charuco_corners, charuco_ids, marker_corners, marker_ids = detector.detectBoard(gray)
        count = 0 if charuco_ids is None else len(charuco_ids)
        found = charuco_ids is not None and charuco_corners is not None and count >= MIN_CHARUCO_CORNERS
        if not found:
            return False, charuco_corners, charuco_ids, marker_corners, marker_ids
        return True, charuco_corners.astype(np.float32), charuco_ids.astype(np.int32), marker_corners, marker_ids

    if hasattr(cv2.aruco, 'ArucoDetector'):
        detector = cv2.aruco.ArucoDetector(dictionary)
        marker_corners, marker_ids, _ = detector.detectMarkers(gray)
    else:
        marker_corners, marker_ids, _ = cv2.aruco.detectMarkers(gray, dictionary)

    if marker_ids is None or len(marker_ids) == 0:
        return False, None, None, marker_corners, marker_ids

    if not hasattr(cv2.aruco, 'interpolateCornersCharuco'):
        raise RuntimeError(
            'This OpenCV build has cv2.aruco but neither CharucoDetector.detectBoard nor '
            'interpolateCornersCharuco. Install opencv-contrib-python in the notebook kernel.'
        )

    count, charuco_corners, charuco_ids = cv2.aruco.interpolateCornersCharuco(
        marker_corners,
        marker_ids,
        gray,
        board,
    )
    count = 0 if count is None else int(count)
    found = charuco_ids is not None and charuco_corners is not None and count >= MIN_CHARUCO_CORNERS
    if not found:
        return False, charuco_corners, charuco_ids, marker_corners, marker_ids
    return True, charuco_corners.astype(np.float32), charuco_ids.astype(np.int32), marker_corners, marker_ids


def find_chessboard_corners(gray: np.ndarray, board_size=BOARD_SIZE):
    if CALIBRATION_TARGET == 'charuco':
        found, corners, _ids, _marker_corners, _marker_ids = detect_charuco_corners(gray)
        return found, corners

    if hasattr(cv2, 'findChessboardCornersSB'):
        found, corners = cv2.findChessboardCornersSB(gray, board_size)
        if found:
            return True, corners.astype(np.float32)

    found, corners = cv2.findChessboardCorners(gray, board_size, None)
    if not found:
        return False, None

    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)
    corners = cv2.cornerSubPix(gray, corners.astype(np.float32), (11, 11), (-1, -1), criteria)
    return True, corners


def object_points_for_charuco_ids(charuco_ids: np.ndarray) -> np.ndarray:
    all_object_corners = get_charuco_object_corners()
    ids = charuco_ids.reshape(-1).astype(int)
    return all_object_corners[ids].reshape(-1, 1, 3).astype(np.float32)


def collect_mono_points(image_dir: Path, board_size=BOARD_SIZE, square_size=SQUARE_SIZE):
    objp = make_object_points(board_size, square_size)
    objpoints, imgpoints, valid_files, rejected = [], [], [], []
    image_size = None

    for path in sorted(image_dir.glob('*.png')):
        gray = read_gray(path)
        if gray is None:
            rejected.append((path.name, 'read failed'))
            continue

        if image_size is None:
            image_size = tuple(int(v) for v in gray.shape[::-1])
        elif image_size != tuple(int(v) for v in gray.shape[::-1]):
            rejected.append((path.name, 'image size mismatch'))
            continue

        if CALIBRATION_TARGET == 'charuco':
            found, corners, ids, _marker_corners, marker_ids = detect_charuco_corners(gray)
            if not found:
                marker_count = 0 if marker_ids is None else len(marker_ids)
                corner_count = 0 if ids is None else len(ids)
                rejected.append((path.name, f'ChArUco not enough corners: markers={marker_count}, corners={corner_count}'))
                continue
            objpoints.append(object_points_for_charuco_ids(ids))
            imgpoints.append(corners)
        else:
            found, corners = find_chessboard_corners(gray, board_size)
            if not found:
                rejected.append((path.name, 'board not found'))
                continue
            objpoints.append(objp.copy())
            imgpoints.append(corners)

        valid_files.append(path.name)

    return objpoints, imgpoints, image_size, valid_files, rejected


def calibrate_single_camera(image_dir: Path, label: str, min_images=MIN_MONO_IMAGES) -> MonoCalibration:
    objpoints, imgpoints, image_size, valid_files, rejected = collect_mono_points(image_dir)
    print(f'{label}: {len(valid_files)} valid image(s), {len(rejected)} rejected')
    if rejected:
        print('First rejected:', rejected[:8])
    if image_size is None or len(valid_files) < min_images:
        raise RuntimeError(f'{label}: need at least {min_images} valid calibration images')

    rms, matrix, distortion, rvecs, tvecs = cv2.calibrateCamera(
        objpoints,
        imgpoints,
        image_size,
        None,
        None,
    )
    print(f'{label} mono RMS: {rms:.4f}px')
    print(f'{label} principal point: ({matrix[0, 2]:.1f}, {matrix[1, 2]:.1f})')
    print(f'{label} max abs distortion: {np.abs(distortion).max():.4f}')
    return MonoCalibration(rms, matrix, distortion, rvecs, tvecs, image_size, valid_files)


def save_pickle(path: Path, data: dict) -> None:
    with open(path, 'wb') as handle:
        pickle.dump(data, handle)
    print(f'Saved {path}')


## Camera Capture Helpers

Run these cells when the Basler cameras are connected. Capture mono images first; capture stereo pairs only after the mono intrinsics have been saved.


In [ ]:
def require_pypylon():
    if pylon is None:
        raise RuntimeError('pypylon is not installed or the Basler SDK is missing')


def list_basler_devices():
    require_pypylon()
    tlf = pylon.TlFactory.GetInstance()
    devices = tlf.EnumerateDevices()
    for idx, device in enumerate(devices):
        print(idx, device.GetFriendlyName(), device.GetSerialNumber())
    return devices


def open_camera(index: int):
    require_pypylon()
    tlf = pylon.TlFactory.GetInstance()
    devices = tlf.EnumerateDevices()
    if index >= len(devices):
        raise RuntimeError(f'Camera index {index} not available; found {len(devices)} camera(s)')
    camera = pylon.InstantCamera(tlf.CreateDevice(devices[index]))
    camera.Open()
    return camera


def make_converter():
    require_pypylon()
    converter = pylon.ImageFormatConverter()
    converter.OutputPixelFormat = pylon.PixelType_BGR8packed
    converter.OutputBitAlignment = pylon.OutputBitAlignment_MsbAligned
    return converter


def capture_one_frame(camera, converter, timeout_ms=5000):
    camera.StartGrabbingMax(1)
    grab = camera.RetrieveResult(timeout_ms, pylon.TimeoutHandling_ThrowException)
    try:
        if not grab.GrabSucceeded():
            raise RuntimeError('Frame grab failed')
        image = converter.Convert(grab)
        return image.GetArray().copy()
    finally:
        grab.Release()


def save_mono_calibration_image(camera_index: int, label: str, out_dir: Path, rotate_180=False):
    camera = open_camera(camera_index)
    converter = make_converter()
    try:
        frame = capture_one_frame(camera, converter)
    finally:
        camera.Close()

    if rotate_180:
        frame = cv2.rotate(frame, cv2.ROTATE_180)
    ts = time.strftime('%Y%m%d_%H%M%S')
    path = out_dir / f'{label}_mono_{ts}.png'
    cv2.imwrite(str(path), frame)
    print(f'Saved {path}')
    return path


def save_stereo_calibration_pair(left_index=LEFT_CAMERA_INDEX, right_index=RIGHT_CAMERA_INDEX):
    cam_left = open_camera(left_index)
    cam_right = open_camera(right_index)
    conv_left = make_converter()
    conv_right = make_converter()
    try:
        frame_left = capture_one_frame(cam_left, conv_left)
        frame_right = capture_one_frame(cam_right, conv_right)
    finally:
        cam_left.Close()
        cam_right.Close()

    if ROTATE_RIGHT_180:
        frame_right = cv2.rotate(frame_right, cv2.ROTATE_180)

    ts = time.strftime('%Y%m%d_%H%M%S')
    left_path = STEREO_CALIB_DIR / f'stereo_calib_{ts}_L.png'
    right_path = STEREO_CALIB_DIR / f'stereo_calib_{ts}_R.png'
    cv2.imwrite(str(left_path), frame_left)
    cv2.imwrite(str(right_path), frame_right)
    print(f'Saved {left_path}')
    print(f'Saved {right_path}')
    return left_path, right_path

def take_stereo_picture():
    cam_left = open_camera(LEFT_CAMERA_INDEX)
    cam_right = open_camera(RIGHT_CAMERA_INDEX)
    conv_left = make_converter()
    conv_right = make_converter()
    try:
        frame_left = capture_one_frame(cam_left, conv_left)
        frame_right = capture_one_frame(cam_right, conv_right)
    finally:
        cam_left.Close()
        cam_right.Close()

    if ROTATE_RIGHT_180:
        frame_right = cv2.rotate(frame_right, cv2.ROTATE_180)
    ts = time.strftime('%Y%m%d_%H%M%S')
    left_path = CAPTURE_DIR / f'stereo_capture_{ts}_L.png'
    right_path = CAPTURE_DIR / f'stereo_capture_{ts}_R.png'
    cv2.imwrite(str(left_path), frame_left)
    cv2.imwrite(str(right_path), frame_right)
    print(f'Saved {left_path}')
    print(f'Saved {right_path}')
    return left_path, right_path

## Capture Mono Calibration Images

Take several sharp images per camera while moving the board through the full image area. Delete blurry or partial captures before running calibration.


In [ ]:
# Uncomment to check camera order.
# list_basler_devices()


In [ ]:
# Run repeatedly after moving the board.
# save_mono_calibration_image(LEFT_CAMERA_INDEX, 'left', MONO_LEFT_DIR)


In [ ]:
# Run repeatedly after moving the board.
# save_mono_calibration_image(RIGHT_CAMERA_INDEX, 'right', MONO_RIGHT_DIR, rotate_180=ROTATE_RIGHT_180)


## Calibrate Cameras Separately

This step writes `mono_camera_params.pkl` with the intrinsic parameters for both cameras.


In [ ]:
left_calib = calibrate_single_camera(MONO_LEFT_DIR, 'left')
right_calib = calibrate_single_camera(MONO_RIGHT_DIR, 'right')

if left_calib.image_size != right_calib.image_size:
    raise RuntimeError(f'Left/right image sizes differ: {left_calib.image_size} vs {right_calib.image_size}')

mono_params = {
    'calibration_target': CALIBRATION_TARGET,
    'board_size': BOARD_SIZE,
    'square_size': SQUARE_SIZE,
    'charuco_squares': CHARUCO_SQUARES,
    'charuco_square_size': CHARUCO_SQUARE_SIZE,
    'charuco_marker_size': CHARUCO_MARKER_SIZE,
    'aruco_dict_name': ARUCO_DICT_NAME,
    'charuco_legacy_pattern': CHARUCO_LEGACY_PATTERN,
    'image_size': left_calib.image_size,
    'left': {
        'rms': left_calib.rms,
        'camera_matrix': left_calib.camera_matrix,
        'distortion': left_calib.distortion,
        'valid_files': left_calib.valid_files,
    },
    'right': {
        'rms': right_calib.rms,
        'camera_matrix': right_calib.camera_matrix,
        'distortion': right_calib.distortion,
        'valid_files': right_calib.valid_files,
    },
}
save_pickle(MONO_PARAMS_FILE, mono_params)

## Mono Undistortion Check

Compare one calibration image per camera before and after undistortion.


In [ ]:
def show_raw_vs_undistorted_mono(sample='middle'):
    if not MONO_PARAMS_FILE.exists():
        raise RuntimeError(f'Missing {MONO_PARAMS_FILE}; run mono calibration first')
    with open(MONO_PARAMS_FILE, 'rb') as handle:
        mono_params = pickle.load(handle)

    def choose_file(directory: Path):
        files = sorted(directory.glob('*.png'))
        if not files:
            raise RuntimeError(f'No images found in {directory}')
        if sample == 'first':
            return files[0]
        if sample == 'last':
            return files[-1]
        if isinstance(sample, int):
            return files[sample]
        return files[len(files) // 2]

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    for row, (label, directory) in enumerate([('left', MONO_LEFT_DIR), ('right', MONO_RIGHT_DIR)]):
        path = choose_file(directory)
        image = cv2.imread(str(path))
        if image is None:
            raise RuntimeError(f'Could not read {path}')
        params = mono_params[label]
        camera_matrix = np.asarray(params['camera_matrix'], dtype=np.float64)
        distortion = np.asarray(params['distortion'], dtype=np.float64)
        undistorted = cv2.undistort(image, camera_matrix, distortion)

        axes[row, 0].imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        axes[row, 0].set_title(f'{label} original\n{path.name}')
        axes[row, 0].axis('off')

        axes[row, 1].imshow(cv2.cvtColor(undistorted, cv2.COLOR_BGR2RGB))
        axes[row, 1].set_title(f'{label} undistorted\nRMS={params["rms"]:.4f}px')
        axes[row, 1].axis('off')

    plt.tight_layout()
    plt.show()


show_raw_vs_undistorted_mono(sample='middle')


## Capture Stereo Calibration Pairs

Do not move the cameras or change their settings after mono calibration. Capture the board in varied positions, distances, and tilts while it remains visible in both cameras.


In [ ]:
# Run repeatedly after moving the board.
# save_stereo_calibration_pair()


## Stereo Calibration

The stereo calibration uses the saved mono intrinsics as fixed camera parameters and writes `stereo_params_mono_first.pkl`.


In [ ]:
def _right_variant_gray(gray: np.ndarray, variant: str) -> np.ndarray:
    if variant == 'as_saved':
        return gray
    if variant == 'rot180':
        return cv2.rotate(gray, cv2.ROTATE_180)
    if variant == 'flip_horizontal':
        return cv2.flip(gray, 1)
    if variant == 'flip_vertical':
        return cv2.flip(gray, 0)
    raise ValueError(f'Unknown right image variant: {variant}')


def _right_variant_bgr(image: np.ndarray, variant: str) -> np.ndarray:
    if variant == 'as_saved':
        return image
    if variant == 'rot180':
        return cv2.rotate(image, cv2.ROTATE_180)
    if variant == 'flip_horizontal':
        return cv2.flip(image, 1)
    if variant == 'flip_vertical':
        return cv2.flip(image, 0)
    raise ValueError(f'Unknown right image variant: {variant}')


def _transform_points_for_variant(points: np.ndarray, image_size: tuple[int, int], variant: str) -> np.ndarray:
    if variant == 'as_saved':
        return points
    width, height = image_size
    transformed = points.copy()
    xy = transformed.reshape(-1, 2)
    if variant == 'flip_horizontal':
        xy[:, 0] = (width - 1) - xy[:, 0]
    elif variant == 'flip_vertical':
        xy[:, 1] = (height - 1) - xy[:, 1]
    elif variant == 'rot180':
        xy[:, 0] = (width - 1) - xy[:, 0]
        xy[:, 1] = (height - 1) - xy[:, 1]
    else:
        raise ValueError(f'Unknown right image variant: {variant}')
    return transformed


def _detect_right_charuco_for_calibration(gray_right_raw: np.ndarray, image_size: tuple[int, int]):
    # ArUco codes are not mirror-invariant. Detect on the saved image first;
    # if a geometric variant is requested, transform detected corner coordinates
    # afterward instead of mirroring marker pixels before detection.
    found, corners, ids, marker_corners, marker_ids = detect_charuco_corners(gray_right_raw)
    if found:
        corners = _transform_points_for_variant(corners, image_size, SELECTED_RIGHT_IMAGE_VARIANT)
        return found, corners, ids, marker_corners, marker_ids, 'detected_as_saved_points_transformed'

    # Fallback for the rare case where the saved image itself is mirrored/rotated
    # relative to a readable marker code orientation.
    if SELECTED_RIGHT_IMAGE_VARIANT != 'as_saved':
        gray_variant = _right_variant_gray(gray_right_raw, SELECTED_RIGHT_IMAGE_VARIANT)
        found_v, corners_v, ids_v, marker_corners_v, marker_ids_v = detect_charuco_corners(gray_variant)
        if found_v:
            return found_v, corners_v, ids_v, marker_corners_v, marker_ids_v, 'detected_after_image_variant'

    return found, corners, ids, marker_corners, marker_ids, 'not_detected'


def _bgr_to_rgb(image: np.ndarray) -> np.ndarray:
    return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)


def _load_mono_params() -> dict:
    if not MONO_PARAMS_FILE.exists():
        raise RuntimeError(f'Missing {MONO_PARAMS_FILE}; run mono calibration first')
    with open(MONO_PARAMS_FILE, 'rb') as handle:
        return pickle.load(handle)


def _mono_intrinsics_for_stereo(mono_params: dict):
    left_key = 'right' if SELECTED_SWAPPED_INTRINSICS else 'left'
    right_key = 'left' if SELECTED_SWAPPED_INTRINSICS else 'right'
    return (
        np.asarray(mono_params[left_key]['camera_matrix'], dtype=np.float64),
        np.asarray(mono_params[left_key]['distortion'], dtype=np.float64),
        np.asarray(mono_params[right_key]['camera_matrix'], dtype=np.float64),
        np.asarray(mono_params[right_key]['distortion'], dtype=np.float64),
        float(mono_params[left_key]['rms']),
        float(mono_params[right_key]['rms']),
        left_key,
        right_key,
    )


def _collect_stereo_calibration_points():
    all_object_corners = get_charuco_object_corners()
    objpoints, imgpoints_left, imgpoints_right = [], [], []
    valid_pairs, rejected_pairs = [], []
    image_size = None

    for left_path in sorted(STEREO_CALIB_DIR.glob('*_L.png')):
        right_path = Path(str(left_path).replace('_L.png', '_R.png'))
        if not right_path.exists():
            rejected_pairs.append((left_path.name, 'missing right image'))
            continue

        gray_left = read_gray(left_path)
        gray_right_raw = read_gray(right_path)
        if gray_left is None or gray_right_raw is None:
            rejected_pairs.append((left_path.name, 'read failed'))
            continue

        pair_size = tuple(int(v) for v in gray_left.shape[::-1])
        if pair_size != tuple(int(v) for v in gray_right_raw.shape[::-1]):
            rejected_pairs.append((left_path.name, 'left/right image size mismatch'))
            continue
        if image_size is None:
            image_size = pair_size
        elif image_size != pair_size:
            rejected_pairs.append((left_path.name, 'mixed image sizes'))
            continue

        found_left, corners_left, ids_left, _marker_corners_l, marker_ids_l = detect_charuco_corners(gray_left)
        found_right, corners_right, ids_right, _marker_corners_r, marker_ids_r, right_detection_mode = _detect_right_charuco_for_calibration(gray_right_raw, pair_size)
        left_markers = 0 if marker_ids_l is None else len(marker_ids_l)
        right_markers = 0 if marker_ids_r is None else len(marker_ids_r)
        left_corners = 0 if ids_left is None else len(ids_left)
        right_corners = 0 if ids_right is None else len(ids_right)

        if not (found_left and found_right):
            rejected_pairs.append(
                (
                    left_path.name,
                    f'detection failed: L markers/corners={left_markers}/{left_corners}, '
                    f'R markers/corners={right_markers}/{right_corners}, mode={right_detection_mode}',
                )
            )
            continue

        left_by_id = {int(idx): corner for idx, corner in zip(ids_left.reshape(-1), corners_left)}
        right_by_id = {int(idx): corner for idx, corner in zip(ids_right.reshape(-1), corners_right)}
        common_ids = sorted(set(left_by_id).intersection(right_by_id))
        if len(common_ids) < MIN_CHARUCO_CORNERS:
            rejected_pairs.append((left_path.name, f'only {len(common_ids)} common ChArUco corners'))
            continue

        objpoints.append(all_object_corners[common_ids].reshape(-1, 1, 3).astype(np.float32))
        imgpoints_left.append(np.asarray([left_by_id[idx] for idx in common_ids], dtype=np.float32).reshape(-1, 1, 2))
        imgpoints_right.append(np.asarray([right_by_id[idx] for idx in common_ids], dtype=np.float32).reshape(-1, 1, 2))
        valid_pairs.append(
            {
                'left': left_path.name,
                'right': right_path.name,
                'common_ids': common_ids,
                'common_corners': len(common_ids),
                'left_markers': left_markers,
                'right_markers': right_markers,
                'right_detection_mode': right_detection_mode,
            }
        )

    return objpoints, imgpoints_left, imgpoints_right, image_size, valid_pairs, rejected_pairs


def _rectified_y_residuals(params: dict, imgpoints_left, imgpoints_right):
    residuals = []
    per_pair = []
    for pair, points_left, points_right in zip(params['valid_pairs'], imgpoints_left, imgpoints_right):
        rect_left = cv2.undistortPoints(points_left, params['mtxL'], params['distL'], R=params['R1'], P=params['P1']).reshape(-1, 2)
        rect_right = cv2.undistortPoints(points_right, params['mtxR'], params['distR'], R=params['R2'], P=params['P2']).reshape(-1, 2)
        ydiff = rect_left[:, 1] - rect_right[:, 1]
        residuals.extend(ydiff.tolist())
        per_pair.append(
            {
                'left': pair['left'],
                'corners': pair['common_corners'],
                'median_abs_y': float(np.median(np.abs(ydiff))),
                'p95_abs_y': float(np.percentile(np.abs(ydiff), 95)),
                'max_abs_y': float(np.max(np.abs(ydiff))),
            }
        )
    residuals = np.asarray(residuals, dtype=np.float64)
    return {
        'median_abs_y': float(np.median(np.abs(residuals))),
        'p95_abs_y': float(np.percentile(np.abs(residuals), 95)),
        'max_abs_y': float(np.max(np.abs(residuals))),
        'per_pair': per_pair,
    }


def _draw_horizontal_lines(image: np.ndarray, line_count=12) -> np.ndarray:
    out = image.copy()
    for y in np.linspace(60, out.shape[0] - 60, line_count, dtype=int):
        cv2.line(out, (0, y), (out.shape[1] - 1, y), (0, 255, 0), 1)
    return out


def _show_uncalibrated_vs_rectified(params: dict, pair_index=-1):
    left_files = sorted(STEREO_CALIB_DIR.glob('*_L.png'))
    if not left_files:
        print('No stereo calibration pairs to show')
        return
    left_path = left_files[pair_index]
    right_path = Path(str(left_path).replace('_L.png', '_R.png'))
    left = cv2.imread(str(left_path))
    right = cv2.imread(str(right_path))
    if left is None or right is None:
        print(f'Could not read {left_path.name} / {right_path.name}')
        return
    right = _right_variant_bgr(right, params['right_image_variant'])

    raw_combined = np.hstack([left, right])
    raw_combined = _draw_horizontal_lines(raw_combined)

    image_size = tuple(params['image_size'])
    map_lx, map_ly = cv2.initUndistortRectifyMap(params['mtxL'], params['distL'], params['R1'], params['P1'], image_size, cv2.CV_32FC1)
    map_rx, map_ry = cv2.initUndistortRectifyMap(params['mtxR'], params['distR'], params['R2'], params['P2'], image_size, cv2.CV_32FC1)
    rect_left = cv2.remap(left, map_lx, map_ly, cv2.INTER_LINEAR)
    rect_right = cv2.remap(right, map_rx, map_ry, cv2.INTER_LINEAR)
    rect_combined = np.hstack([rect_left, rect_right])
    rect_combined = _draw_horizontal_lines(rect_combined)

    fig, axes = plt.subplots(2, 1, figsize=(15, 12))
    axes[0].imshow(_bgr_to_rgb(raw_combined))
    axes[0].set_title(f'Uncalibrated pair with guide lines\n{left_path.name} / {right_path.name}')
    axes[0].axis('off')
    axes[1].imshow(_bgr_to_rgb(rect_combined))
    axes[1].set_title('Calibrated + rectified pair with guide lines')
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()


def run_fixed_intrinsics_stereo_calibration(show_pair_index=-1):
    mono_params = _load_mono_params()
    mtx_left, dist_left, mtx_right, dist_right, mono_rms_left, mono_rms_right, mono_left_source, mono_right_source = _mono_intrinsics_for_stereo(mono_params)
    objpoints, imgpoints_left, imgpoints_right, image_size, valid_pairs, rejected_pairs = _collect_stereo_calibration_points()

    print('Stereo pair collection')
    print(f'  valid pairs: {len(valid_pairs)}')
    print(f'  rejected pairs: {len(rejected_pairs)}')
    print(f'  image size: {image_size}')
    if valid_pairs:
        counts = np.asarray([pair['common_corners'] for pair in valid_pairs])
        print(f'  common ChArUco corners min/median/max: {counts.min()} / {np.median(counts):.0f} / {counts.max()}')
    if rejected_pairs:
        print('  first rejected pairs:')
        for item in rejected_pairs[:8]:
            print('   ', item)
    if image_size is None or len(valid_pairs) < MIN_STEREO_PAIRS:
        raise RuntimeError(f'Need at least {MIN_STEREO_PAIRS} valid stereo pairs, found {len(valid_pairs)}')
    if image_size != tuple(mono_params['image_size']):
        raise RuntimeError(f'Stereo image size {image_size} does not match mono image size {mono_params["image_size"]}')

    stereo_rms, _, _, _, _, R, T, E, F = cv2.stereoCalibrate(
        objpoints,
        imgpoints_left,
        imgpoints_right,
        mtx_left,
        dist_left,
        mtx_right,
        dist_right,
        image_size,
        criteria=(cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 100, 1e-5),
        flags=cv2.CALIB_FIX_INTRINSIC,
    )
    R1, R2, P1, P2, Q, roi1, roi2 = cv2.stereoRectify(
        mtx_left,
        dist_left,
        mtx_right,
        dist_right,
        image_size,
        R,
        T,
        flags=cv2.CALIB_ZERO_DISPARITY,
        alpha=1,
    )

    params = {
        'calibration_target': CALIBRATION_TARGET,
        'charuco_squares': CHARUCO_SQUARES,
        'charuco_square_size': CHARUCO_SQUARE_SIZE,
        'charuco_marker_size': CHARUCO_MARKER_SIZE,
        'charuco_legacy_pattern': CHARUCO_LEGACY_PATTERN,
        'aruco_dict_name': ARUCO_DICT_NAME,
        'image_size': image_size,
        'mono_rms_left': mono_rms_left,
        'mono_rms_right': mono_rms_right,
        'mono_left_source': mono_left_source,
        'mono_right_source': mono_right_source,
        'stereo_rms': float(stereo_rms),
        'mtxL': mtx_left,
        'distL': dist_left,
        'mtxR': mtx_right,
        'distR': dist_right,
        'R': R,
        'T': T,
        'E': E,
        'F': F,
        'R1': R1,
        'R2': R2,
        'P1': P1,
        'P2': P2,
        'Q': Q,
        'roi1': roi1,
        'roi2': roi2,
        'valid_pairs': valid_pairs,
        'rejected_pairs': rejected_pairs,
        'right_image_variant': SELECTED_RIGHT_IMAGE_VARIANT,
        'swapped_intrinsics': SELECTED_SWAPPED_INTRINSICS,
    }
    params['rectified_y_residuals'] = _rectified_y_residuals(params, imgpoints_left, imgpoints_right)
    save_pickle(STEREO_PARAMS_FILE, params)

    T_flat = T.reshape(-1)
    residuals = params['rectified_y_residuals']
    print('\nStereo calibration result')
    print(f'  mono RMS left/right: {mono_rms_left:.4f}px / {mono_rms_right:.4f}px')
    print(f'  stereo RMS: {stereo_rms:.4f}px')
    print(f'  T: {T_flat}')
    print(f'  baseline norm: {np.linalg.norm(T):.3f} mm')
    print(f'  |Tz| / baseline: {abs(T_flat[2]) / max(np.linalg.norm(T), 1e-9):.3f}')
    print(
        '  rectified y residual median/p95/max: '
        f"{residuals['median_abs_y']:.3f} / {residuals['p95_abs_y']:.3f} / {residuals['max_abs_y']:.3f}px"
    )
    print('  worst pairs by rectified p95 y residual:')
    for row in sorted(residuals['per_pair'], key=lambda item: item['p95_abs_y'], reverse=True)[:8]:
        print(f"    {row['left']}: corners={row['corners']}, median={row['median_abs_y']:.3f}px, p95={row['p95_abs_y']:.3f}px")

    _show_uncalibrated_vs_rectified(params, pair_index=show_pair_index)
    return params


stereo_params = run_fixed_intrinsics_stereo_calibration(show_pair_index=-1)


## Capture Scene Pair For Placement Check

Use this cell to capture a normal stereo scene pair after calibration.


In [ ]:
# Capture one scene pair for the placement overlay.
# save_scene_stereo_pair()


## Scene Placement Overlay

The green area is computed for the selected working-distance range. It marks rectified pixels that can have a corresponding pixel in the other camera image. Adjust `WORKSPACE_DEPTH_MIN_MM` and `WORKSPACE_DEPTH_MAX_MM` for the expected object distance.


In [ ]:
WORKSPACE_DEPTH_MIN_MM = 400.0
WORKSPACE_DEPTH_MAX_MM = 600.0


def _load_stereo_params(path=STEREO_PARAMS_FILE):
    if not Path(path).exists():
        raise RuntimeError(f'Missing {path}; run stereo calibration first')
    with open(path, 'rb') as handle:
        return pickle.load(handle)


def _rectify_pair_for_display(left_bgr, right_bgr, params):
    right_bgr = _right_variant_bgr(right_bgr, params.get('right_image_variant', SELECTED_RIGHT_IMAGE_VARIANT))
    image_size = tuple(params['image_size'])
    map_lx, map_ly = cv2.initUndistortRectifyMap(params['mtxL'], params['distL'], params['R1'], params['P1'], image_size, cv2.CV_32FC1)
    map_rx, map_ry = cv2.initUndistortRectifyMap(params['mtxR'], params['distR'], params['R2'], params['P2'], image_size, cv2.CV_32FC1)
    rect_left = cv2.remap(left_bgr, map_lx, map_ly, cv2.INTER_LINEAR)
    rect_right = cv2.remap(right_bgr, map_rx, map_ry, cv2.INTER_LINEAR)

    src_mask = np.full((image_size[1], image_size[0]), 255, dtype=np.uint8)
    valid_left = cv2.remap(src_mask, map_lx, map_ly, cv2.INTER_NEAREST, borderValue=0) > 0
    valid_right = cv2.remap(src_mask, map_rx, map_ry, cv2.INTER_NEAREST, borderValue=0) > 0
    return rect_left, rect_right, valid_left, valid_right


def _draw_epipolar_lines(image: np.ndarray, line_count=12) -> np.ndarray:
    out = image.copy()
    for y in np.linspace(60, out.shape[0] - 60, line_count, dtype=int):
        cv2.line(out, (0, y), (out.shape[1] - 1, y), (0, 255, 0), 1)
    return out


def _tint_mask(image: np.ndarray, mask: np.ndarray, color=(0, 240, 0)) -> np.ndarray:
    # Use neutral grayscale outside the mask so the camera's cyan/green color cast
    # cannot be mistaken for the placement mask.
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    out = cv2.cvtColor((gray * 0.18).astype(np.uint8), cv2.COLOR_GRAY2BGR)
    inside = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
    tint = np.zeros_like(image)
    tint[:, :] = color
    inside[mask] = cv2.addWeighted(inside[mask], 0.35, tint[mask], 0.65, 0)
    out[mask] = inside[mask]

    mask_u8 = mask.astype(np.uint8) * 255
    contours, _ = cv2.findContours(mask_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(out, contours, -1, (255, 255, 255), 2)
    return out


def _binary_mask_view(mask: np.ndarray) -> np.ndarray:
    out = np.zeros((mask.shape[0], mask.shape[1], 3), dtype=np.uint8)
    out[mask] = (0, 220, 0)
    return out


def _disparity_range_from_depth(params: dict, depth_min_mm: float, depth_max_mm: float):
    if depth_min_mm <= 0 or depth_max_mm <= 0:
        raise ValueError('Depth values must be positive millimeters')
    near = min(depth_min_mm, depth_max_mm)
    far = max(depth_min_mm, depth_max_mm)
    focal_px = float(params['P1'][0, 0])
    baseline_mm = abs(float(params['P2'][0, 3]) / max(abs(float(params['P2'][0, 0])), 1e-9))
    min_disp = int(np.floor(focal_px * baseline_mm / far))
    max_disp = int(np.ceil(focal_px * baseline_mm / near))
    min_disp = max(0, min_disp)
    max_disp = max(min_disp + 1, max_disp)
    return min_disp, max_disp, focal_px, baseline_mm


def _mask_lookup_with_offset(mask: np.ndarray, offset: int) -> np.ndarray:
    # Shift the mask horizontally by the expected disparity.
    out = np.zeros_like(mask, dtype=bool)
    if offset == 0:
        out[:] = mask
    elif offset > 0:
        out[:, :-offset] = mask[:, offset:]
    else:
        out[:, -offset:] = mask[:, :offset]
    return out


def _matchable_masks_from_disparity(valid_left, valid_right, min_disp: int, max_disp: int, params: dict):
    # For rectified stereo, corresponding points are on the same row.
    # The x shift is the disparity. With OpenCV's projection matrices, the sign
    # of P2[0, 3] tells us whether right_x = left_x - disparity or + disparity.
    p2_shift = float(params['P2'][0, 3])
    disparity_sign = -1 if p2_shift < 0 else 1

    matchable_left = np.zeros_like(valid_left, dtype=bool)
    matchable_right = np.zeros_like(valid_right, dtype=bool)
    for disparity in range(min_disp, max_disp + 1):
        left_to_right_offset = disparity_sign * disparity
        matchable_left |= _mask_lookup_with_offset(valid_right, left_to_right_offset)
        matchable_right |= _mask_lookup_with_offset(valid_left, -left_to_right_offset)

    return valid_left & matchable_left, valid_right & matchable_right, disparity_sign


def _resolve_stereo_pair_folder(pair_folder=None):
    candidates = []
    if pair_folder is not None:
        requested = Path(pair_folder)
        candidates.append(requested)
        if not requested.is_absolute() and requested.name == 'captures':
            candidates.append(CAPTURE_DIR)
            candidates.append(PROJECT_ROOT / requested)
            candidates.append(Path('..') / requested)
    candidates.append(CAPTURE_DIR)
    candidates.append(Path('captures'))
    candidates.append(Path('..') / 'captures')

    seen = set()
    unique_candidates = []
    for candidate in candidates:
        key = str(candidate)
        if key not in seen:
            seen.add(key)
            unique_candidates.append(candidate)

    for candidate in unique_candidates:
        if candidate.exists() and any(candidate.glob('*_L.png')):
            return candidate
    checked = ', '.join(str(candidate) for candidate in unique_candidates)
    raise RuntimeError(f'No stereo pairs found. Checked: {checked}')


def show_scene_placement_overlay(
    pair_folder=None,
    pair_index=-1,
    depth_min_mm=WORKSPACE_DEPTH_MIN_MM,
    depth_max_mm=WORKSPACE_DEPTH_MAX_MM,
):
    params = _load_stereo_params()
    folder = _resolve_stereo_pair_folder(pair_folder)
    left_files = sorted(folder.glob('*_L.png'))

    left_path = left_files[pair_index]
    right_path = Path(str(left_path).replace('_L.png', '_R.png'))
    if not right_path.exists():
        raise RuntimeError(f'Missing right image for {left_path.name}')

    left = cv2.imread(str(left_path))
    right = cv2.imread(str(right_path))
    if left is None or right is None:
        raise RuntimeError(f'Could not read {left_path.name} / {right_path.name}')

    display_right = _right_variant_bgr(right, params.get('right_image_variant', SELECTED_RIGHT_IMAGE_VARIANT))
    rect_left, rect_right, valid_left, valid_right = _rectify_pair_for_display(left, right, params)
    min_disp, max_disp, focal_px, baseline_mm = _disparity_range_from_depth(params, depth_min_mm, depth_max_mm)
    matchable_left, matchable_right, disparity_sign = _matchable_masks_from_disparity(valid_left, valid_right, min_disp, max_disp, params)

    original_side_by_side = np.hstack([left, display_right])
    rectified_side_by_side = np.hstack([
        _draw_epipolar_lines(rect_left),
        _draw_epipolar_lines(rect_right),
    ])
    matchable_side_by_side = np.hstack([
        _draw_epipolar_lines(_tint_mask(rect_left, matchable_left, color=(0, 240, 0))),
        _draw_epipolar_lines(_tint_mask(rect_right, matchable_right, color=(0, 240, 0))),
    ])
    binary_mask_side_by_side = np.hstack([
        _draw_epipolar_lines(_binary_mask_view(matchable_left)),
        _draw_epipolar_lines(_binary_mask_view(matchable_right)),
    ])

    print('Placement mask parameters')
    print(f'  expected depth range: {min(depth_min_mm, depth_max_mm):.0f}..{max(depth_min_mm, depth_max_mm):.0f} mm')
    print(f'  focal length: {focal_px:.1f} px')
    print(f'  rectified baseline: {baseline_mm:.1f} mm')
    print(f'  expected disparity range: {min_disp}..{max_disp} px')
    print(f'  right_x = left_x {disparity_sign:+d} disparity')
    print(f'  green coverage left/right: {100 * matchable_left.mean():.1f}% / {100 * matchable_right.mean():.1f}%')

    fig, axes = plt.subplots(4, 1, figsize=(15, 20))
    axes[0].imshow(cv2.cvtColor(original_side_by_side, cv2.COLOR_BGR2RGB))
    axes[0].set_title(f'Original pair side by side: {left_path.name} / {right_path.name}')
    axes[0].axis('off')

    axes[1].imshow(cv2.cvtColor(rectified_side_by_side, cv2.COLOR_BGR2RGB))
    axes[1].set_title('Rectified pair side by side. Matching points should lie on the same horizontal green line.')
    axes[1].axis('off')

    axes[2].imshow(cv2.cvtColor(matchable_side_by_side, cv2.COLOR_BGR2RGB))
    axes[2].set_title('Placement overlay for selected depth range: green = projections that can have a counterpart in the other view')
    axes[2].axis('off')

    axes[3].imshow(cv2.cvtColor(binary_mask_side_by_side, cv2.COLOR_BGR2RGB))
    axes[3].set_title('Binary placement mask for the selected depth range. Object must be green in each image, usually at different x positions.')
    axes[3].axis('off')
    plt.tight_layout()
    plt.show()


# Set the expected working distance range before running this cell.
show_scene_placement_overlay(pair_folder=CAPTURE_DIR, pair_index=-1)
